# Task 3 — log target check (v5)

The slide suggests log transformation of the target. Test one controlled change to the selected two-stage/continuous mixture: fit the positive-only CatBoost MAE model on `log1p(spending_30d)` and invert with `expm1`. Hold the occurrence gate, 0.50 cutoff, continuous blend, and 80/20 mixture fixed. Use the same five development folds and report MAE in THB, which preserves ranking under the slide's monotone NMAE formula on a fixed validation set. The official Kaggle calculation remains unconfirmed. Do not use Test or leaderboard scores.

In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task3.models.spending_models import PositiveGate,WeightedPair
SEED=set_seed()
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split,StratifiedKFold
SRC=ROOT/'src';TASK=SRC/'task3';SAVE=TASK/'save';RES=TASK/'models'/'results'
DATA='eda_combined_v1';NAME='t3_log_positive_amount_cv5_v5'
card=json.loads((TASK/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/card['splits']['train']['path'])
y=train.spending_30d.astype(float)
bands=pd.cut(y,bins=[-1,0,100,1000,10000,50000,np.inf],labels=False)
dev_idx,hold_idx=train_test_split(np.arange(len(y)),test_size=.20,stratify=bands,random_state=SEED)
dev=train.iloc[dev_idx].reset_index(drop=True);hold=train.iloc[hold_idx]
dev_bands=bands.iloc[dev_idx].reset_index(drop=True)
folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev,dev_bands))
features=[c for c in dev if c not in ['player_id','spending_30d']]
old=joblib.load(SAVE/'t3_gate_blend_cv5_v2.joblib')
assert old.first_weight==.8 and old.first_model.threshold==.5 and len(features)==48
print(len(dev),'development rows',len(hold),'reused holdout rows')

83200 development rows 20800 reused holdout rows


## 1. Rebuild existing mixture from saved fold predictions

In [2]:
parts=[]
for i,(_,check) in enumerate(folds,1):
    gate=pd.read_parquet(RES/f'round3_gate_fold{i}_v1.parquet')
    blend=pd.read_parquet(RES/f'round3_blend_fold{i}_v1.parquet')
    assert gate.row_pos.tolist()==check.tolist()==blend.row_pos.tolist()
    assert gate.true.equals(blend.true)
    continuous=np.maximum(old.second_model.catboost_weight*blend.cat+
         (1-old.second_model.catboost_weight)*blend.lgb,0)
    baseline=.8*np.where(gate.probability>=.5,np.maximum(gate.positive_amount,0),0)+.2*continuous
    parts.append(pd.DataFrame({'fold':i,'row_pos':check,'true':gate.true,
         'probability':gate.probability,'continuous':continuous,'baseline':baseline}))
base=pd.concat(parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
assert np.array_equal(base.row_pos,np.arange(len(dev)))
assert np.array_equal(base.true,dev.spending_30d)
def fold_mae(pred):
    return np.array([mean_absolute_error(base.loc[base.fold==i,'true'],pred[base.fold==i]) for i in range(1,6)])
old_scores=fold_mae(base.baseline.to_numpy())
assert abs(old_scores.mean()-2895.92)<.2
print('Baseline fold MAE:',old_scores.round(2),'mean',old_scores.mean())

Baseline fold MAE: [2815.58 2973.69 2797.07 2929.51 2963.76] mean 2895.921540742373


## 2. Fit log-target amount model, save, and evaluate

Each fold fits the saved positive-only pipeline's preprocessing on positive training rows only. The log transform applies to training labels; predictions are returned in THB. Only the positive amount component changes.

In [3]:
paths=[SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',SAVE/f'{NAME}-evaluation.json']
def amount_model():
    return TransformedTargetRegressor(regressor=clone(old.first_model.positive_amount_model),
        func=np.log1p,inverse_func=np.expm1,check_inverse=False)
key={'dataset':DATA,'created_at':card['created_at'],'seed':SEED,
     'source':'t3_gate_blend_cv5_v2','change':'log1p positive amount target'}
parts=[]
for i,(fit,check) in enumerate(folds,1):
    path=RES/f'{NAME}-fold{i}.parquet';meta=path.with_suffix('.json')
    if path.exists()!=meta.exists():raise RuntimeError(f'Incomplete fold {i}')
    if path.exists():
        assert json.loads(meta.read_text(encoding='utf-8'))=={'key':key,'fold':i}
        part=pd.read_parquet(path)
    else:
        if any(p.exists() for p in paths):raise RuntimeError('Saved model has missing OOF fold')
        positive=fit[dev.iloc[fit].spending_30d.to_numpy()>0]
        m=amount_model();m.fit(dev.iloc[positive][features],dev.iloc[positive].spending_30d)
        pred=np.maximum(np.asarray(m.predict(dev.iloc[check][features]),dtype=float),0)
        part=pd.DataFrame({'fold':i,'row_pos':check,'amount':pred})
        part.to_parquet(path,index=False)
        meta.write_text(json.dumps({'key':key,'fold':i},indent=2),encoding='utf-8')
        print('Fold',i,'fitted',flush=True)
    assert np.array_equal(part.row_pos,check)
    parts.append(part)
amount_oof=pd.concat(parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
assert np.array_equal(amount_oof.row_pos,np.arange(len(dev)))
new_pred=.8*np.where(base.probability>=.5,amount_oof.amount,0)+.2*base.continuous
new_scores=fold_mae(new_pred.to_numpy())
if not any(p.exists() for p in paths):
    positive=dev.spending_30d>0
    amount=amount_model();amount.fit(dev.loc[positive,features],dev.loc[positive,'spending_30d'])
    model=WeightedPair(PositiveGate(old.first_model.occurrence_model,amount,.5),old.second_model,.8)
    hold_pred=model.predict(hold[features])
    save_model(task=3,name=NAME,model=model,data_name=DATA,
        card={'idea':'Log1p positive-amount target in fixed two-stage mixture',
              'source_model':'t3_gate_blend_cv5_v2','seed':SEED,
              'development_cv_mean_MAE_thb':float(new_scores.mean())})
    evaluate_model(task=3,name=NAME,score=float(mean_absolute_error(hold.spending_30d,hold_pred)),
        metric_name='MAE_thb',higher_is_better=False,
        card={'split':'reused 20% holdout','development_fold_MAE_thb':new_scores.tolist(),
              'same_fold_baseline_MAE_thb':old_scores.tolist(),
              'note':'MAE in THB; official Kaggle NMAE implementation unconfirmed'})
else:
    assert all(p.exists() for p in paths)
    result=json.loads(paths[2].read_text(encoding='utf-8'))
    model=joblib.load(paths[0]);hold_pred=model.predict(hold[features])
    assert abs(result['score']-mean_absolute_error(hold.spending_30d,hold_pred))<.05
display(pd.DataFrame({'baseline':old_scores,'log_amount':new_scores,'delta':new_scores-old_scores},index=range(1,6)).round(2))
print('CV MAE delta:',(new_scores-old_scores).mean(),'wins:',int((new_scores<old_scores).sum()),'/5')
print('Reused holdout MAE:',mean_absolute_error(hold.spending_30d,hold_pred))

Fold 1 fitted


Fold 2 fitted


Fold 3 fitted


Fold 4 fitted


Fold 5 fitted


,baseline,log_amount,delta
1,2815.58,2743.14,-72.44
2,2973.69,2886.62,-87.07
3,2797.07,2711.28,-85.79
4,2929.51,2848.55,-80.96
5,2963.76,2885.25,-78.51


CV MAE delta: -80.9535283807384 wins: 5 /5
Reused holdout MAE: 2780.9410106141663


## Decision

Prefer the previously selected mixture unless the log-target variant has a meaningful, consistent paired-fold MAE improvement. A lower MAE implies a higher score under the slide's formula on the same validation set, but it cannot establish a public Kaggle gain.